In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import polars as pl
import lightgbm as lgb
from sklearn.model_selection import GroupKFold

DATA_DIR = Path('/kaggle/input/competitions/optiver-realized-volatility-prediction')

def compute_rmspe(y_true, y_pred):
    return np.sqrt(np.mean(np.square((y_true - y_pred) / y_true)))


In [2]:
def compute_stock_features(book_path: Path, trade_path: Path = None) -> pl.DataFrame:
    stock_id = int(book_path.stem.split('=')[-1])

    book_df = (
        pl.read_parquet(book_path)
        .with_columns([
            (
                (pl.col('bid_price1') * pl.col('ask_size1') + pl.col('ask_price1') * pl.col('bid_size1'))
                / (pl.col('bid_size1') + pl.col('ask_size1'))
            ).alias('wap1'),
            (
                (pl.col('bid_price2') * pl.col('ask_size2') + pl.col('ask_price2') * pl.col('bid_size2'))
                / (pl.col('bid_size2') + pl.col('ask_size2'))
            ).alias('wap2'),
            (pl.col('ask_price1') - pl.col('bid_price1')).alias('spread1'),
            (pl.col('ask_price2') - pl.col('bid_price2')).alias('spread2'),
            (
                (pl.col('bid_size1') - pl.col('ask_size1'))
                / (pl.col('bid_size1') + pl.col('ask_size1'))
            ).alias('obi1'),
            (pl.col('bid_size1') + pl.col('ask_size1') + pl.col('bid_size2') + pl.col('ask_size2')).alias('total_depth'),
        ])
        .with_columns([
            (pl.col('wap1').log() - pl.col('wap1').log().shift(1)).over('time_id').alias('log_ret1'),
            (pl.col('wap2').log() - pl.col('wap2').log().shift(1)).over('time_id').alias('log_ret2'),
        ])
    )

    book_aggs = [
        (pl.col('log_ret1').pow(2).sum().sqrt()).alias('vol_wap1_600'),
        (pl.col('log_ret2').pow(2).sum().sqrt()).alias('vol_wap2_600'),
        (pl.col('spread1').mean()).alias('mean_spread1_600'),
        (pl.col('spread2').mean()).alias('mean_spread2_600'),
        (pl.col('obi1').mean()).alias('mean_obi1_600'),
        (pl.col('total_depth').mean()).alias('mean_depth_600'),
        (pl.col('log_ret1').filter(pl.col('seconds_in_bucket') >= 300).pow(2).sum().sqrt()).alias('vol_wap1_300'),
        (pl.col('log_ret2').filter(pl.col('seconds_in_bucket') >= 300).pow(2).sum().sqrt()).alias('vol_wap2_300'),
        (pl.col('log_ret1').filter(pl.col('seconds_in_bucket') >= 450).pow(2).sum().sqrt()).alias('vol_wap1_150'),
        (pl.col('log_ret2').filter(pl.col('seconds_in_bucket') >= 450).pow(2).sum().sqrt()).alias('vol_wap2_150'),
    ]

    stock_df = book_df.group_by('time_id').agg(book_aggs).with_columns(pl.lit(stock_id).alias('stock_id'))

    if trade_path and trade_path.exists():
        trade_df = (
            pl.read_parquet(trade_path)
            .with_columns([
                (pl.col('price').log() - pl.col('price').log().shift(1)).over('time_id').alias('trade_log_ret')
            ])
            .group_by('time_id')
            .agg([
                (pl.col('trade_log_ret').pow(2).sum().sqrt()).alias('trade_vol_600'),
                (pl.col('size').sum()).alias('trade_size_sum'),
                (pl.col('order_count').sum()).alias('trade_order_count_sum'),
            ])
        )
        stock_df = stock_df.join(trade_df, on='time_id', how='left')

    return stock_df

In [3]:
train_book_paths = sorted(Path(f'{DATA_DIR}/book_train.parquet').glob('stock_id=*'))
train_trade_dir = Path(f'{DATA_DIR}/trade_train.parquet')

print(f"Extracting features across {len(train_book_paths)} stock partitions...")
feature_dfs = []
for p in train_book_paths:
    stock_id = p.stem.split('=')[-1]
    t_path = train_trade_dir / f'stock_id={stock_id}'
    feature_dfs.append(compute_stock_features(p, t_path))

full_features = (
    pl.concat(feature_dfs)
    .fill_null(0.0)
    .with_columns([
        pl.col('stock_id').cast(pl.Int64),
        pl.col('time_id').cast(pl.Int64)
    ])
)

# Market-wide aggregations across stocks per time_id
market_aggs = (
    full_features
    .group_by('time_id')
    .agg([
        pl.col('vol_wap1_600').mean().alias('market_mean_vol_600'),
        pl.col('vol_wap1_600').std().alias('market_std_vol_600'),
        pl.col('vol_wap1_300').mean().alias('market_mean_vol_300'),
        pl.col('mean_spread1_600').mean().alias('market_mean_spread_600'),
    ])
    .with_columns(pl.col('time_id').cast(pl.Int64))
)

train_labels = (
    pl.read_csv(f'{DATA_DIR}/train.csv')
    .with_columns([
        pl.col('stock_id').cast(pl.Int64),
        pl.col('time_id').cast(pl.Int64)
    ])
)

train_matrix = (
    train_labels
    .join(full_features, on=['stock_id', 'time_id'], how='inner')
    .join(market_aggs, on='time_id', how='left')
    .with_columns([
        (pl.col('vol_wap1_600') / (pl.col('market_mean_vol_600') + 1e-8)).alias('vol_to_market_ratio'),
        (pl.col('vol_wap1_600') - pl.col('market_mean_vol_600')).alias('vol_market_diff'),
    ])
    .fill_null(0.0)
)

print(f"Train matrix ready: {train_matrix.shape}")
print(f"Total features extracted: {len(train_matrix.columns) - 3}")

Extracting features across 112 stock partitions...
Train matrix ready: (428932, 22)
Total features extracted: 19


In [4]:
feature_cols = [c for c in train_matrix.columns if c not in ['stock_id', 'time_id', 'target', 'row_id']]
X = train_matrix.select(feature_cols).to_pandas()
y = train_matrix['target'].to_numpy()
groups = train_matrix['time_id'].to_numpy()

gkf = GroupKFold(n_splits=5)
oof_predictions = np.zeros(len(train_matrix))
models = []

params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 1,
    'verbosity': -1,
    'random_state': 42,
    'n_jobs': -1
}

for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups)):
    X_train, y_train = X.iloc[train_idx], y[train_idx]
    X_val, y_val = X.iloc[val_idx], y[val_idx]

    train_data = lgb.Dataset(X_train, label=y_train, weight=1.0 / (y_train ** 2))
    val_data = lgb.Dataset(X_val, label=y_val, weight=1.0 / (y_val ** 2), reference=train_data)

    model = lgb.train(
        params,
        train_data,
        num_boost_round=1000,
        valid_sets=[val_data],
        callbacks=[lgb.early_stopping(50, verbose=False)]
    )

    oof_predictions[val_idx] = model.predict(X_val)
    models.append(model)
    print(f"Fold {fold + 1} RMSPE: {compute_rmspe(y_val, oof_predictions[val_idx]):.5f}")

cv_rmspe = compute_rmspe(y, oof_predictions)
print(f"\nOverall Out-of-Fold RMSPE: {cv_rmspe:.5f}")

Fold 1 RMSPE: 0.22936
Fold 2 RMSPE: 0.23416
Fold 3 RMSPE: 0.23119
Fold 4 RMSPE: 0.22974
Fold 5 RMSPE: 0.23099

Overall Out-of-Fold RMSPE: 0.23109


In [5]:
test_book_paths = sorted(Path(f'{DATA_DIR}/book_test.parquet').glob('stock_id=*'))
test_trade_dir = Path(f'{DATA_DIR}/trade_test.parquet')

test_feature_dfs = []
for p in test_book_paths:
    stock_id = p.stem.split('=')[-1]
    t_path = test_trade_dir / f'stock_id={stock_id}'
    test_feature_dfs.append(compute_stock_features(p, t_path))

test_features = (
    pl.concat(test_feature_dfs)
    .fill_null(0.0)
    .with_columns([
        pl.col('stock_id').cast(pl.Int64),
        pl.col('time_id').cast(pl.Int64)
    ])
)

test_market_aggs = (
    test_features
    .group_by('time_id')
    .agg([
        pl.col('vol_wap1_600').mean().alias('market_mean_vol_600'),
        pl.col('vol_wap1_600').std().alias('market_std_vol_600'),
        pl.col('vol_wap1_300').mean().alias('market_mean_vol_300'),
        pl.col('mean_spread1_600').mean().alias('market_mean_spread_600'),
    ])
    .with_columns(pl.col('time_id').cast(pl.Int64))
)

test_meta = (
    pl.read_csv(f'{DATA_DIR}/test.csv')
    .with_columns([
        pl.col('stock_id').cast(pl.Int64),
        pl.col('time_id').cast(pl.Int64)
    ])
)

test_matrix = (
    test_meta
    .join(test_features, on=['stock_id', 'time_id'], how='left')
    .join(test_market_aggs, on='time_id', how='left')
    .with_columns([
        (pl.col('vol_wap1_600') / (pl.col('market_mean_vol_600') + 1e-8)).alias('vol_to_market_ratio'),
        (pl.col('vol_wap1_600') - pl.col('market_mean_vol_600')).alias('vol_market_diff'),
    ])
    .fill_null(0.0)
)

for c in feature_cols:
    if c not in test_matrix.columns:
        test_matrix = test_matrix.with_columns(pl.lit(0.0).alias(c))

X_test = test_matrix.select(feature_cols).to_pandas()
ensemble_preds = np.mean([m.predict(X_test) for m in models], axis=0)

submission = pl.DataFrame({
    'row_id': test_matrix['row_id'],
    'target': ensemble_preds
})

submission.write_csv('submission.csv')
print("Saved submission.csv successfully!")

Saved submission.csv successfully!
